# Data Cleaning

Import libraries:

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

Local paths for yukon_salmon.csv and monthly_20261030T1856.csv:

In [15]:
salmon_path = r"C:\Users\portl\OneDrive\Documents\Data Science Certificate\DATA 5100 FQ\Projects\Salmon\data\yukon_salmon.csv"
water_path = r"C:\Users\portl\OneDrive\Documents\Data Science Certificate\DATA 5100 FQ\Projects\Salmon\data\monthly_20261030T1856.csv"

In [35]:
salmon_df = pd.read_csv(salmon_path)
water_df = pd.read_csv(water_path, on_bad_lines = 'skip', skiprows = 1)

A big span of rows in the water data have an extra comma and no data, so they were skipped to avoid the read error. The first row contained explanatory information and was excluded also.

Inspect salmon_df:

In [11]:
salmon_df.head()

,Chapter,Region,Species,Year,Spawners,Harvest,Harvest Rate,Returns,Data Source
0,8,Yukon,Chinook,1982,35777,126408,0.779406,162185,YUKON JTC (25)-01 Appendix B11
1,8,Yukon,Chinook,1983,55250,127886,0.698312,183136,YUKON JTC (25)-01 Appendix B11
2,8,Yukon,Chinook,1984,68331,69967,0.505915,138298,YUKON JTC (25)-01 Appendix B11
3,8,Yukon,Chinook,1985,39082,119290,0.753227,158372,YUKON JTC (25)-01 Appendix B11
4,8,Yukon,Chinook,1986,49804,126371,0.717304,176175,YUKON JTC (25)-01 Appendix B11


In [12]:
salmon_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 70 entries, 0 to 69
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Chapter       70 non-null     int64  
 1   Region        70 non-null     str    
 2   Species       70 non-null     str    
 3   Year          70 non-null     int64  
 4   Spawners      70 non-null     int64  
 5   Harvest       70 non-null     int64  
 6   Harvest Rate  70 non-null     float64
 7   Returns       70 non-null     int64  
 8   Data Source   70 non-null     str    
dtypes: float64(1), int64(5), str(3)
memory usage: 8.2 KB


Exclude 'Chapter', 'Region', and 'Data Source' columns:

In [13]:
salmon_df = salmon_df[["Species", "Year", "Spawners", "Harvest", "Harvest Rate", "Returns"]]

In [14]:
salmon_df.tail()

,Species,Year,Spawners,Harvest,Harvest Rate,Returns
65,Chum,2020,23512,1488,0.059520,25000
66,Chum,2021,23170,0,0.000000,23000
67,Chum,2022,22034,0,0.000000,22000
68,Chum,2023,22090,910,0.039565,23000
69,Chum,2024,16174,0,0.000000,16000


Inspect water_df:

In [36]:
water_df.head()

,ID,PARAM,MM--YYYY,Value
0,09AB001,1,10--1943,483.0
1,09AB001,1,11--1943,298.0
2,09AB001,1,12--1943,200.0
3,09AB001,1,01--1944,109.0
4,09AB001,1,02--1944,68.4


In [37]:
water_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1875 entries, 0 to 1874
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0    ID       1875 non-null   str    
 1   PARAM     1875 non-null   int64  
 2   MM--YYYY  1875 non-null   str    
 3   Value     1469 non-null   float64
dtypes: float64(1), int64(1), str(2)
memory usage: 86.2 KB


In [38]:
water_df.columns

Index([' ID', 'PARAM', 'MM--YYYY', 'Value'], dtype='str')

Note that the ID column actually has a space in front

In [39]:
water_df[' ID'].nunique()

1

As expected, this is data from the Whitehorse station only. So we can safely drop this column.

In [40]:
water_df = water_df.drop(columns = ' ID')

In [41]:
water_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1875 entries, 0 to 1874
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   PARAM     1875 non-null   int64  
 1   MM--YYYY  1875 non-null   str    
 2   Value     1469 non-null   float64
dtypes: float64(1), int64(1), str(1)
memory usage: 58.7 KB


Let's also change the MM--YYYY column to a datetime type:

In [42]:
water_df['MM--YYYY'] = pd.to_datetime(water_df['MM--YYYY'], format = '%m--%Y')

In [43]:
water_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1875 entries, 0 to 1874
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   PARAM     1875 non-null   int64         
 1   MM--YYYY  1875 non-null   datetime64[us]
 2   Value     1469 non-null   float64       
dtypes: datetime64[us](1), float64(1), int64(1)
memory usage: 44.1 KB


This dataset contains information on the monthly mean discharge (labeled under param 1) as well as mean monthly water level (labeled under param 2).

I think the simplest way to get all of our information lined up together by date would be to separate the values by their param label into two different dataframes, and then merge them back together on MM--YYYY.

In [47]:
flow_df = water_df[water_df['PARAM'] == 1].copy()

In [48]:
flow_df.head()

,PARAM,MM--YYYY,Value
0,1,1943-10-01,483.0
1,1,1943-11-01,298.0
2,1,1943-12-01,200.0
3,1,1944-01-01,109.0
4,1,1944-02-01,68.4


In [49]:
level_df = water_df[water_df['PARAM'] == 2].copy()

In [50]:
level_df.head()

,PARAM,MM--YYYY,Value
891,2,1902-01-01,NaN
892,2,1902-02-01,NaN
893,2,1902-03-01,NaN
894,2,1902-04-01,NaN
895,2,1902-05-01,NaN


We'll do an outer merge to keep all the information:

In [51]:
water_clean = pd.merge(flow_df, level_df, on = 'MM--YYYY', how = 'outer')

In [52]:
water_clean.head()

,PARAM_x,MM--YYYY,Value_x,PARAM_y,Value_y
0,NaN,1902-01-01,NaN,2.0,NaN
1,NaN,1902-02-01,NaN,2.0,NaN
2,NaN,1902-03-01,NaN,2.0,NaN
3,NaN,1902-04-01,NaN,2.0,NaN
4,NaN,1902-05-01,NaN,2.0,NaN


In [54]:
water_clean.tail()

,PARAM_x,MM--YYYY,Value_x,PARAM_y,Value_y
1471,NaN,2024-08-01,NaN,2.0,30.573
1472,NaN,2024-09-01,NaN,2.0,30.261
1473,NaN,2024-10-01,NaN,2.0,29.612
1474,NaN,2024-11-01,NaN,2.0,29.623
1475,NaN,2024-12-01,NaN,2.0,30.816


In [53]:
water_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 1476 entries, 0 to 1475
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   PARAM_x   891 non-null    float64       
 1   MM--YYYY  1476 non-null   datetime64[us]
 2   Value_x   886 non-null    float64       
 3   PARAM_y   984 non-null    float64       
 4   Value_y   583 non-null    float64       
dtypes: datetime64[us](1), float64(4)
memory usage: 57.8 KB


Now we need to reconfigure and relabel our columns so that rows will consist of the date (MM--YYYY), flow (Value_x) (if available), and level (Value_y) (if available).

In [55]:
water_clean = water_clean.drop(columns = ['PARAM_x', 'PARAM_y'])

In [62]:
water_clean = water_clean.rename(columns = {'MM--YYYY' : 'date', 'Value_x' : 'flow', 'Value_y' : 'level'})

In [63]:
water_clean

,date,flow,level
0,1902-01-01,NaN,NaN
1,1902-02-01,NaN,NaN
2,1902-03-01,NaN,NaN
3,1902-04-01,NaN,NaN
4,1902-05-01,NaN,NaN
...,...,...,...
1471,2024-08-01,NaN,30.573
1472,2024-09-01,NaN,30.261
1473,2024-10-01,NaN,29.612
1474,2024-11-01,NaN,29.623


Because our salmon data has to do with salmon travelling upriver to spawn, we should select our watershed data pertaining to that timeframe.

According to Yukon.ca, Chinook salmon pass through Whitehorse in July and August. (https://yukon.ca/en/outdoor-recreation-and-wildlife/yukon-wildlife/chinook-salmon)

According to the Yukon River Panel, Chinook salmon travel through the upper river region in July, and spawn in the Yukon River drainage in July and September (https://www.yukonriverpanel.com/about-us/yukon-river-panel/yukon-river-salmon/chinook/)

According to the Yukon River Panel, Chum salmon travel through the upper river region in September, and spawning occurs in October and November. (https://www.yukonriverpanel.com/about-us/yukon-river-panel/yukon-river-salmon/chum/)

Let's select the broadest range and choose flow and level data between June and November.

In [64]:
salmon_df['Year']

0     1982
1     1983
2     1984
3     1985
4     1986
      ... 
65    2020
66    2021
67    2022
68    2023
69    2024
Name: Year, Length: 70, dtype: int64

Our salmon data range is also only from 1982 to 2024, so we'll only select water data from those years as well.

In [67]:
water_select_year = water_clean[water_clean['date'].between('1982-01-01', '2024-12-31')]

In [68]:
water_select_year

,date,flow,level
960,1982-01-01,144.0,NaN
961,1982-02-01,154.0,NaN
962,1982-03-01,173.0,NaN
963,1982-04-01,158.0,NaN
964,1982-05-01,150.0,NaN
...,...,...,...
1471,2024-08-01,NaN,30.573
1472,2024-09-01,NaN,30.261
1473,2024-10-01,NaN,29.612
1474,2024-11-01,NaN,29.623


In [70]:
water_select_months = water_select_year[water_select_year['date'].dt.month.isin([6, 7, 8, 9, 10, 11])]

In [71]:
water_select_months

,date,flow,level
965,1982-06-01,264.0,NaN
966,1982-07-01,432.0,NaN
967,1982-08-01,472.0,NaN
968,1982-09-01,463.0,NaN
969,1982-10-01,289.0,NaN
...,...,...,...
1470,2024-07-01,NaN,30.201
1471,2024-08-01,NaN,30.573
1472,2024-09-01,NaN,30.261
1473,2024-10-01,NaN,29.612


In [72]:
water_select_months.info()

<class 'pandas.DataFrame'>
Index: 258 entries, 965 to 1474
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    258 non-null    datetime64[us]
 1   flow    215 non-null    float64       
 2   level   58 non-null     float64       
dtypes: datetime64[us](1), float64(2)
memory usage: 8.1 KB


As a quick shortcut, the website where we retrieved the water data shows where data is available; either partial (P) or complete (C)(https://wateroffice.ec.gc.ca/report/data_availability_e.html?type=historical&station=09AB001&parameter_type=Flow+and+Level)

And we can see that the level data ends in 1972 and doesn't start again until October of 2014, which is why we only have 58 observations left over.

A refresher of our data ranges:

Salmon: 1982-2024 (single measure per year)

In [73]:
salmon_df['Year'].agg(['min', 'max'])

min    1982
max    2024
Name: Year, dtype: int64

Temp: 2011-2022 (multiple measures per day)

In [ ]:
#temp data not loaded here

Flow: approximately 1943 to 2017 (measured once per month?)

In [75]:
flow_df['MM--YYYY'].agg(['min', 'max'])

min   1943-10-01
max   2017-12-01
Name: MM--YYYY, dtype: datetime64[us]

Level: effectively 2014 to 2024 (some measures in the early 1900s, more robust measures from 1928 to 1972, a gap until 2014, and then pretty consistent to 2024)

In [77]:
level_df['MM--YYYY'].agg(['min', 'max'])

min   1902-01-01
max   2024-12-01
Name: MM--YYYY, dtype: datetime64[us]

### We could make two models:

One with predictor variables of temp and level for 2014 to 2022 (richer data, narrower timeframe)

And one with predictor variables of flow only from 1982 to 2017 (shallower data, wider timeframe)

The only place all of our data overlaps is from 2014 (level starts) to 2017 (flow ends), which would only give us 5 years to work with.

### Exporting final dataframes:

Once we figure out how we want to process our data (means, medians, date ranges, etc.), we'll create and export dataframes that will let us do that.